# Channel Selection by SNR

**Dataset**: MOABB BNCI2014-001 (Motor Imagery)  
**Channels**: 22 channels  
**Sampling rate**: 250 Hz  
**Subject**: 1

---

## Overview

We compute SNR for each channel and select the highest quality ones.

## Expected outputs

- Bar chart of channels sorted by SNR
- Topomap showing selected (green) vs rejected (red) channels

## Key parameters

| Parameter | Value |
| --- | --- |
| N_SELECT | 10 |
| window | 50 |


## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB).


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


## 3. Explore the data


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


## 4. Compute SNR and select channels


In [ ]:
N_SELECT = 10

def compute_snr(signal):
    signal_var = np.var(signal)
    window = 50
    moving_avg = np.convolve(signal, np.ones(window) / window, mode='same')
    noise = signal - moving_avg
    noise_var = np.var(noise)
    if noise_var == 0:
        return 0.0
    return signal_var / noise_var

snr_values = np.zeros(n_channels)
for ch in range(n_channels):
    snr_list = [compute_snr(X[trial, ch, :]) for trial in range(n_trials)]
    snr_values[ch] = np.mean(snr_list)

sorted_idx = np.argsort(snr_values)[::-1]
selected = sorted_idx[:N_SELECT]
print(f'Selected channels: {[ch_names[i] for i in selected]}')


## 5. Interactive plot

**What to look for:**

- Selected channels concentrate in central and parietal regions
- Frontal and temporal channels are usually rejected


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=('SNR ranking', 'Topomap'))
colors = ['green' if i in selected else 'gray' for i in sorted_idx]
fig.add_trace(go.Bar(x=[ch_names[i] for i in sorted_idx], y=snr_values[sorted_idx], marker_color=colors, name='SNR'), row=1, col=1)
fig.update_xaxes(tickangle=90, row=1, col=1)

data = dataset.get_data(subjects=[1])
raw = data[1][list(data[1].keys())[0]][list(data[1][list(data[1].keys())[0]].keys())[0]]
montage = raw.get_montage()
ch_pos = montage.get_positions()['ch_pos']
ch_names_all = [ch for ch in raw.ch_names if ch in ch_pos and not ch.startswith('EOG')]
positions = np.array([ch_pos[ch] for ch in ch_names_all])
pos_2d = positions[:, :2]
scale = 1.0 / np.max(np.abs(pos_2d))
pos_2d = pos_2d * scale * 0.95

for i in range(n_channels):
    color = 'green' if i in selected else 'red'
    fig.add_trace(go.Scatter(x=[pos_2d[i, 0]], y=[pos_2d[i, 1]], mode='markers+text', text=[ch_names_all[i]], textposition='top right', marker=dict(color=color, size=10), showlegend=False), row=1, col=2)
fig.update_layout(height=500, title_text='Channel Selection by SNR')
fig.show()


## What did we learn?

- SNR measures channel quality as signal-to-noise ratio
- Selected channels concentrate in the central region
- A fast method that does not require model training
